In [ ]:
setwd("./real_data_EAS")

In [3]:
library(MRAPSS)
library(data.table)
library(MASS)
library(ggpubr)

basdir = getwd()

### format TPMI data

In [50]:
pheno.info = read.csv("./raw_data/TPMI/summary.csv")
pheno.info$EUR.phecode = as.character(pheno.info$EUR.phecode)

In [31]:
# Read in dbSNP153 hg38 data
dbSNP38 <- readr::read_delim("./raw_data/TPMI/dbSnp153Common.bed", col_names = FALSE)[, c(1:4)]
colnames(dbSNP38) <- c("CHR", "START", "END", "rsID")

setDT(dbSNP38)

setkey(dbSNP38, CHR, END)

Rows: 15160915 Columns: 17
── Column specification ─────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): X1, X4, X5, X7, X10, X11, X12, X14, X15
dbl (8): X2, X3, X6, X8, X9, X13, X16, X17

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [32]:
head(pheno.info)

,X,Category,Phenotype,EAS.phecode,EAS.samples,EAS.cases,EAS.controls,EAS.address,EAS.data.link,EUR.phecode
,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>
1,1,Metabolic,Low-density lipoprotein cholesterol,LDL-C,165202,165202,0,LDL-C.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/LDL-C,30780
2,2,Protein,Hemoglobin,HB,194120,194120,0,HB.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/HB,30020
3,3,Metabolic,Glycated hemoglobin,HbA1c,90509,90509,0,HbA1c.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/HbA1c,30750
4,4,Blood pressure,Diastolic blood pressure,DBP,196215,196215,0,DBP.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/DBP,4079
5,5,Anthropometric,Body mass index,BMI,191458,191458,0,BMI.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/BMI,21001
6,6,Liver-related,Serum glutamate pyruvate transaminase,ALT,203766,203766,0,ALT.saige.txt.gz,https://pheweb.ibms.sinica.edu.tw/pheno/ALT,30620


In [ ]:
for(i in 1:dim(pheno.info)[1]){
    
    phecode =  pheno.info[['EAS.phecode']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    tw_address = paste0('./raw_data/TPMI/', pheno.info[['EAS.address']][i])
    data_tw = fread(tw_address)
    
    ### assign rsID
    # some with the suffix 'chr'; some not
    if(nchar(data_tw$CHR[1]) < 3){
        data_tw$CHR_modified = paste0("chr", data_tw$CHR)
    }else{
        data_tw$CHR_modified = data_tw$CHR
    }
    
    # match with keys; assign 'rsID' to data_tw in column 'SNPID'
    data_tw$SNPID = data_tw$MarkerID
    data_tw[dbSNP38, on = .(CHR_modified = CHR, POS = END), SNPID := i.rsID]

    data_tw[is.na(data_tw$SNPID)]$SNPID = data_tw[is.na(data_tw$SNPID)]$MarkerID
    data_tw[(data_tw$SNPID=='NA')]$SNPID = data_tw[(data_tw$SNPID=='NA')]$MarkerID
    
    
    result <- try({
        
        dat_tw = MRAPSS::format_data(data_tw,
                                      snp_col = "SNPID",
                                      b_col = "BETA",
                                      se_col = "SE",
                                      A1_col = "Allele2", ### effect allele
                                      A2_col = "Allele1", ### non-effect allele
                                      p_col = "p.value",
                                      freq_col = "AF_Allele2",
                                      n = pheno.info[['EAS.samples']][i],
                                      min_freq = 0.01)

        ########################### name rule: 'TW_' + its EUR.phecode ############################
        write.table(dat_tw, file=paste0("./formatted_data/TW_", pheno.info[['EUR.phecode']][i]), quote = F, col.names = T, row.names = F)

    }, silent = TRUE)

    if (inherits(result, "try-error")) {
        print(paste("Error in round", i, "- skipping to next round."))
        next
    }
    
}

### format BBJ data

#### exposure traits

In [73]:
file_path <- './raw_data/BBJ/EUR_BBJ_exposure_description.csv'
data_summary_exp <- read.csv(file_path)

In [74]:
data_summary_exp

Category,UKB.Phenotype,UKB.samples,UKB.cases,UKB.controls,UKB.code,UKB.address,BBJ.Phenotype,BBJ.samples,BBJ.cases,BBJ.controls,BBJ.address,BBJ.code
<chr>,<chr>,<int>,<chr>,<chr>,<int>,<lgl>,<chr>,<int>,<chr>,<chr>,<lgl>,<chr>
Anthropometric,Body mass index (BMI),359983,-,-,21001,NA,Body mass index (BMI),163835,-,-,NA,BMI
Blood cell,Hemoglobin (Hb),350474,-,-,30020,NA,Hemoglobin (Hb),152447,-,-,NA,Hb
Blood pressure,Diastolic blood pressure (DBP),340162,-,-,4079,NA,Diastolic blood pressure (DBP),145515,-,-,NA,DBP
Liver-related,Alanine aminotransferase (ALT),344136,-,-,30620,NA,Alanine aminotransferase (ALT),150545,-,-,NA,ALT
Metabolic,Glucose (BS),314916,-,-,30740,NA,Glucose (BS),133336,-,-,NA,Glucose
Metabolic,HbA1c (HbA1c),344182,-,-,30750,NA,HbA1c (HbA1c),71221,-,-,NA,HbA1c
Metabolic,LDL-cholesterol (LDL-C),343621,-,-,30780,NA,LDL-cholesterol (LDL-C),72866,-,-,NA,LDLC


In [ ]:
for(i in 1:dim(data_summary_exp)[1]){
    
    phecode =  data_summary_exp[['BBJ.code']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    BBJ_address = paste0("./raw_data/BBJ/hum0197.v3.BBJ.", phecode, ".v1/GWASsummary_", phecode, "_Japanese_SakaueKanai2020.auto.txt.gz")
    
    data_BBJ = fread(BBJ_address)
    
    result <- try({
        
        dat_BBJ = format_data(data_BBJ,
                              snp_col = "SNP",
                              b_col = "BETA",
                              se_col = "SE",
                              A1_col = "ALLELE1", ### effect allele
                              A2_col = "ALLELE0", ### non-effect allele
                              p_col = "P_BOLT_LMM_INF",
                              freq_col = "A1FREQ",
                              n = data_summary_exp[['BBJ.samples']][i],
                              min_freq = 0.01)

        ########################### name rule: 'BBJ_' + its EUR.phecode ############################
        write.table(dat_BBJ, file=paste0("./formatted_data/BBJ_", data_summary_exp[['UKB.code']][i]), quote = F, col.names = T, row.names = F)

    }, silent = TRUE)

    if (inherits(result, "try-error")) {
        print(paste("Error in round", i, "- skipping to next round."))
        next
    }
    
}

#### outcome traits

In [110]:
data_summary_out = read.csv('./raw_data/BBJ/EUR_BBJ_outcome_description.csv')
data_summary_out$UKB.phecode = as.character(data_summary_out$UKB.phecode)
data_summary_out$phecode_unique = as.character(data_summary_out$phecode_unique)

In [111]:
head(data_summary_out)

,Category,UKB.Phenotype,UKB.phecode,FG.endpoint,UKB.samples,UKB.cases,UKB.controls,UKB.address,BBJ.Phenotype,BBJ.samples,BBJ.cases,BBJ.controls,BBJ.address,phecode_unique
,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
1,ICD10 C,Breast cancer,174.1,C3_BREAST,257730,17389,240341,hum0197.v3.EUR.BC.v1/GWASsummary_BrC_EUR_SakaueKanai2020.auto.txt.gz,Breast cancer,79550,6325,73225,hum0197.v3.BBJ.BC.v1/GWASsummary_BrC_Japanese_SakaueKanai2020.auto.txt.gz,174.1
2,ICD10 C,Prostate cancer,185,C3_PROSTATE,211227,11599,199628,hum0197.v3.EUR.PrC.v1/GWASsummary_PrC_EUR_SakaueKanai2020.auto.txt.gz,Prostate cancer,90332,5672,84660,hum0197.v3.BBJ.PrC.v1/GWASsummary_PrC_Japanese_SakaueKanai2020.auto.txt.gz,185
3,ICD10 D,Iron deficiency anemia,280.1,D3_ANAEMIA_IRONDEF,480941,12317,468624,hum0197.v3.EUR.IDA.v1/GWASsummary_Iron_Deficiency_Anemia_EUR_SakaueKanai2020.auto.txt.gz,Iron deficiency anemia,178675,4618,174057,hum0197.v3.BBJ.IDA.v1/GWASsummary_Iron_Deficiency_Anemia_Japanese_SakaueKanai2020.auto.txt.gz,280.1
4,ICD10 D,Uterine fibroid,218.1,CD2_BENIGN_LEIOMYOMA_UTERI,258718,21024,237694,hum0197.v3.EUR.UF.v1/GWASsummary_UF_EUR_SakaueKanai2020.auto.txt.gz,Uterine fibroid,80208,14475,65733,hum0197.v3.BBJ.UF.v1/GWASsummary_UF_Japanese_SakaueKanai2020.auto.txt.gz,218.1
5,ICD10 G,Epilepsy,345.1,G6_EPLEPSY,458310,4382,453928,hum0197.v3.EUR.Ep.v1/GWASsummary_Epilepsy_EUR_SakaueKanai2020.auto.txt.gz,Epilepsy,178254,2466,175788,hum0197.v3.BBJ.Ep.v1/GWASsummary_Epilepsy_Japanese_SakaueKanai2020.auto.txt.gz,345.1
6,ICD10 H,Cataract,366,H7_CATARACTSENILE,491877,39519,452358,hum0197.v3.EUR.Cat.v1/GWASsummary_Cataract_EUR_SakaueKanai2020.auto.txt.gz,Cataract,178726,38194,140532,hum0197.v3.BBJ.Cat.v1/GWASsummary_Cataract_Japanese_SakaueKanai2020.auto.txt.gz,366


In [ ]:
for(i in 1:dim(data_summary_out)[1]){
    phecode =  data_summary_out[['phecode_unique']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    BBJ_address = paste0("./raw_data/BBJ/", data_summary_out[['BBJ.address']][i])

    data_BBJ = data.table::fread(BBJ_address)
    
    result <- try({
    
        dat_BBJ = format_data(data_BBJ,
                              snp_col = "SNPID",
                              b_col = "BETA",
                              se_col = "SE",
                              A1_col = "Allele2", ### effect allele
                              A2_col = "Allele1", ### non-effect allele
                              p_col = "p.value",
                              freq_col = "AF_Allele2",
                              n = data_summary_out[['BBJ.samples']][i],
                              min_freq = 0.01)
        
        ########################### name rule: 'BBJ_' + its EUR.phecode ############################
        write.table(dat_BBJ, file=paste0("./formatted_data/BBJ_", data_summary_out[['UKB.phecode']][i]), quote = F, col.names = T, row.names = F)

        
    }, silent = TRUE)

    if (inherits(result, "try-error")) {
        print(paste("Error in round", i, "- skipping to next round."))
        next
    }
    
}

### format EUR data

#### exposure traits

In [138]:
variants = fread('./raw_data/EUR/variants.tsv')

In [137]:
file_path <- './raw_data/BBJ/EUR_BBJ_exposure_description.csv'
data_summary_exp <- read.csv(file_path)
data_summary_exp

Category,UKB.Phenotype,UKB.samples,UKB.cases,UKB.controls,UKB.code,UKB.address,BBJ.Phenotype,BBJ.samples,BBJ.cases,BBJ.controls,BBJ.address,BBJ.code
<chr>,<chr>,<int>,<chr>,<chr>,<int>,<lgl>,<chr>,<int>,<chr>,<chr>,<lgl>,<chr>
Anthropometric,Body mass index (BMI),359983,-,-,21001,NA,Body mass index (BMI),163835,-,-,NA,BMI
Blood cell,Hemoglobin (Hb),350474,-,-,30020,NA,Hemoglobin (Hb),152447,-,-,NA,Hb
Blood pressure,Diastolic blood pressure (DBP),340162,-,-,4079,NA,Diastolic blood pressure (DBP),145515,-,-,NA,DBP
Liver-related,Alanine aminotransferase (ALT),344136,-,-,30620,NA,Alanine aminotransferase (ALT),150545,-,-,NA,ALT
Metabolic,Glucose (BS),314916,-,-,30740,NA,Glucose (BS),133336,-,-,NA,Glucose
Metabolic,HbA1c (HbA1c),344182,-,-,30750,NA,HbA1c (HbA1c),71221,-,-,NA,HbA1c
Metabolic,LDL-cholesterol (LDL-C),343621,-,-,30780,NA,LDL-cholesterol (LDL-C),72866,-,-,NA,LDLC


In [ ]:
for(i in 1:dim(data_summary_exp)[1]){
    
    phecode = data_summary_exp[['UKB.code']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    UKB_address = paste0("./raw_data/EUR/", phecode, ".tsv")
    
    data_UKB = fread(UKB_address)
    
    check1 = sum(data_UKB$variant != variants$variant)
    
    if(check1 != 0){
        print('Need more careful alignment with variants.tsv... skip')
        next
    }
    
    data_UKB$SNP = variants$rsid
    
    data_UKB$A1 = variants$alt # effect allele
    data_UKB$A2 = variants$ref # non-effect allele
    
    result <- try({
    
        dat_UKB = format_data(data_UKB,
                      snp_col = "SNP",
                      b_col = "beta",
                      se_col = "se",
                      A1_col = "A1",
                      A2_col = "A2",
                      p_col = "pval",
                      freq_col = "minor_AF",
                      #n = data_summary_exp[['UKB.samples']][i],
                      n_col = 'n_complete_samples',
                      min_freq = 0.01)
        
        write.table(dat_UKB, file=paste0("./formatted_data/EUR_",phecode), quote = F, col.names = T, row.names = F)
        
    }, silent = TRUE)

    if (inherits(result, "try-error")) {
        print(paste("Error in round", i, "- skipping to next round."))
        next
    }
    
}

#### outcome traits

In [153]:
data_summary_out = read.csv('./raw_data/BBJ/EUR_BBJ_outcome_description.csv')
data_summary_out$UKB.phecode = as.character(data_summary_out$UKB.phecode)
data_summary_out$phecode_unique = as.character(data_summary_out$phecode_unique)

In [177]:
data_summary_out

Category,UKB.Phenotype,UKB.phecode,FG.endpoint,UKB.samples,UKB.cases,UKB.controls,UKB.address,BBJ.Phenotype,BBJ.samples,BBJ.cases,BBJ.controls,BBJ.address,phecode_unique
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
ICD10 C,Breast cancer,174.1,C3_BREAST,257730,17389,240341,hum0197.v3.EUR.BC.v1/GWASsummary_BrC_EUR_SakaueKanai2020.auto.txt.gz,Breast cancer,79550,6325,73225,hum0197.v3.BBJ.BC.v1/GWASsummary_BrC_Japanese_SakaueKanai2020.auto.txt.gz,174.1
ICD10 C,Prostate cancer,185,C3_PROSTATE,211227,11599,199628,hum0197.v3.EUR.PrC.v1/GWASsummary_PrC_EUR_SakaueKanai2020.auto.txt.gz,Prostate cancer,90332,5672,84660,hum0197.v3.BBJ.PrC.v1/GWASsummary_PrC_Japanese_SakaueKanai2020.auto.txt.gz,185
ICD10 D,Iron deficiency anemia,280.1,D3_ANAEMIA_IRONDEF,480941,12317,468624,hum0197.v3.EUR.IDA.v1/GWASsummary_Iron_Deficiency_Anemia_EUR_SakaueKanai2020.auto.txt.gz,Iron deficiency anemia,178675,4618,174057,hum0197.v3.BBJ.IDA.v1/GWASsummary_Iron_Deficiency_Anemia_Japanese_SakaueKanai2020.auto.txt.gz,280.1
ICD10 D,Uterine fibroid,218.1,CD2_BENIGN_LEIOMYOMA_UTERI,258718,21024,237694,hum0197.v3.EUR.UF.v1/GWASsummary_UF_EUR_SakaueKanai2020.auto.txt.gz,Uterine fibroid,80208,14475,65733,hum0197.v3.BBJ.UF.v1/GWASsummary_UF_Japanese_SakaueKanai2020.auto.txt.gz,218.1
ICD10 G,Epilepsy,345.1,G6_EPLEPSY,458310,4382,453928,hum0197.v3.EUR.Ep.v1/GWASsummary_Epilepsy_EUR_SakaueKanai2020.auto.txt.gz,Epilepsy,178254,2466,175788,hum0197.v3.BBJ.Ep.v1/GWASsummary_Epilepsy_Japanese_SakaueKanai2020.auto.txt.gz,345.1
ICD10 H,Cataract,366,H7_CATARACTSENILE,491877,39519,452358,hum0197.v3.EUR.Cat.v1/GWASsummary_Cataract_EUR_SakaueKanai2020.auto.txt.gz,Cataract,178726,38194,140532,hum0197.v3.BBJ.Cat.v1/GWASsummary_Cataract_Japanese_SakaueKanai2020.auto.txt.gz,366
ICD10 I,Angina pectoris,411.3,I9_ANGINA,470931,30025,440906,hum0197.v3.EUR.Ang.v1/GWASsummary_Angina_EUR_SakaueKanai2020.auto.txt.gz,Angina pectoris,159165,14007,145158,hum0197.v3.BBJ.Ang.v1/GWASsummary_Angina_Japanese_SakaueKanai2020.auto.txt.gz,411.3
ICD10 I,Myocardial infarction,411.2,I9_MI,461823,20917,440906,hum0197.v3.EUR.MI.v1/GWASsummary_MI_EUR_SakaueKanai2020.auto.txt.gz,Myocardial infarction,161206,14992,146214,hum0197.v3.BBJ.MI.v1/GWASsummary_MI_Japanese_SakaueKanai2020.auto.txt.gz,411.2
ICD10 I,Peripheral arterial disease,443.9,I9_PAD,483078,7114,475964,hum0197.v3.EUR.PAD.v1/GWASsummary_PAD_EUR_SakaueKanai2020.auto.txt.gz,Peripheral arterial disease,177713,4112,173601,hum0197.v3.BBJ.PAD.v1/GWASsummary_PAD_Japanese_SakaueKanai2020.auto.txt.gz,443.9


In [154]:
# Read in dbSNP153 data
dbSNP <- readr::read_delim("./raw_data/EUR/dbSnp153Common.hg19.bed", col_names = FALSE)[, c(1:4)]
colnames(dbSNP) <- c("CHR", "START", "END", "rsID")

setDT(dbSNP)

setkey(dbSNP, CHR, END)

Rows: 14908375 Columns: 17
── Column specification ─────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): X1, X4, X5, X7, X10, X11, X12, X14, X15
dbl (8): X2, X3, X6, X8, X9, X13, X16, X17

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
for(i in 1:dim(data_summary_out)[1]){
    phecode =  data_summary_out[['phecode_unique']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    data_UKB = data.table::fread(paste0('./raw_data/EUR/', data_summary_out[['UKB.address']][i]))
    
    result <- try({
        
        data_UKB$CHR_modified = paste0("chr", data_UKB$CHR)
        data_UKB$SNPID = data_UKB$v

        # match with keys; assign 'rsID' to data_tw in column 'SNPID'
        data_UKB[dbSNP, on = .(CHR_modified = CHR, POS = END), SNPID := i.rsID]

        data_UKB[is.na(data_UKB$SNPID)]$SNPID = data_UKB[is.na(data_UKB$SNPID)]$v
        data_UKB[(data_UKB$SNPID=='NA')]$SNPID = data_UKB[(data_UKB$SNPID=='NA')]$v

        data_UKB$AF_Allele2 = data_UKB$AF_Allele2_UKB
        data_UKB[is.na(data_UKB$AF_Allele2)]$AF_Allele2 = data_UKB[is.na(data_UKB$AF_Allele2)]$AF_Allele2_FG
        
        data_UKB = format_data(data_UKB,
                              snp_col = "SNPID",
                              b_col = "BETA",
                              se_col = "SE",
                              A1_col = "Allele2", # effect allele
                              A2_col = "Allele1", # effect allele
                              p_col = "p.value",
                              freq_col = "AF_Allele2",
                              n = data_summary_out[['UKB.samples']][i],
                              min_freq = 0.01)

        write.table(data_UKB, file=paste0("./formatted_data/EUR_",phecode), quote = F, col.names = T, row.names = F)

    }, silent = TRUE)

    if (inherits(result, "try-error")) {
        print(paste("Error in round", i, "- skipping to next round."))
        next
    }
    
}